# 3-Fold Cross-Validation: Max Pooling Multi-Head Model

Training on QEvasion train split only.
Uses **Max Pooling** with sliding window chunking (stride=256).

In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentence piece matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
import zipfile
from collections import Counter
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

## Configuration

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
N_FOLDS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

## Load Dataset

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_df = ds['train'].to_pandas()

print(f"Train size: {len(train_df)}")
print(f"\nClarity label distribution:")
print(train_df['clarity_label'].value_counts())
print(f"\nEvasion label distribution:")
print(train_df['evasion_label'].value_counts())

## Tokenization

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,  
        padding=False,
        max_length=None,
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    
    return tokenized

# Tokenize training data
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_tokenized = train_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=train_dataset.column_names
)

print(f"Train tokenized: {len(train_tokenized)} samples")

## Data Collator

In [ ]:
@dataclass
class HierarchicalMultiHeadDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor(
            [f.pop("clarity_labels") for f in features], 
            dtype=torch.long
        )
        evasion_labels = torch.tensor(
            [f.pop("evasion_labels") for f in features], 
            dtype=torch.long
        )
        
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = HierarchicalMultiHeadDataCollator(tokenizer=tokenizer)

## Model Architecture: Max Pooling

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadHierarchicalMaxPooling(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        """Override to prevent transformers from trying to inspect notebook source."""
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout 
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)

        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = [] 
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]

            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
  
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]  

            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]  
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)

        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss_clarity = loss_fct(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=None,
            attentions=None,
        )

## Metrics & Trainer

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels

    valid_evasion_mask = labels_evasion !=100

    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')

    if valid_evasion_mask.sum() > 0:
        accuracy_evasion = accuracy_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask])
        f1_evasion = f1_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask], average='macro')
    else:
        accuracy_evasion = 0.0
        f1_evasion = 0.0
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }


class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())

        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)


class EarlyStoppingCallback(TrainerCallback):    
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

# 3-Fold Stratified Cross-Validation with Comprehensive Analysis

In [ ]:
# Create output directory for confusion matrices
os.makedirs("./confusion_matrices_3fold", exist_ok=True)

print("="*60)
print("3-FOLD STRATIFIED CROSS-VALIDATION")
print("="*60)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for out-of-fold predictions
oof_preds_clarity = np.zeros(len(train_df), dtype=int)
oof_preds_evasion = np.zeros(len(train_df), dtype=int)
oof_true_clarity = np.zeros(len(train_df), dtype=int)
oof_true_evasion = np.zeros(len(train_df), dtype=int)

fold_metrics = []

print(f"\nStarting {N_FOLDS}-fold cross-validation...")
print(f"Total training samples: {len(train_df)}\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['clarity_label']), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    fold_train_dataset = train_tokenized.select(train_idx)
    fold_val_dataset = train_tokenized.select(val_idx)
    
    config = AutoConfig.from_pretrained(MODEL_NAME)
    model = MultiHeadHierarchicalMaxPooling(config)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model.roberta.load_state_dict(base_model.state_dict())
    del base_model
    
    training_args = TrainingArguments(
        output_dir=f"./results_multihead_maxpool_3fold_{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8, 
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_combined",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        report_to="none",
    )
    
    trainer = MultiHeadTrainer(
        model=model,
        args=training_args,
        train_dataset=fold_train_dataset,
        eval_dataset=fold_val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
    )

    trainer.train()

    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy_clarity': val_results['eval_accuracy_clarity'],
        'val_f1_clarity': val_results['eval_f1_clarity'],
        'val_accuracy_evasion': val_results['eval_accuracy_evasion'],
        'val_f1_evasion': val_results['eval_f1_evasion'],
        'val_f1_combined': val_results['eval_f1_combined'],
    })
    
    print(f"\nFold {fold} Results:")
    print(f"  Clarity - Accuracy: {val_results['eval_accuracy_clarity']:.4f}, F1: {val_results['eval_f1_clarity']:.4f}")
    print(f"  Evasion - Accuracy: {val_results['eval_accuracy_evasion']:.4f}, F1: {val_results['eval_f1_evasion']:.4f}")
    print(f"  Combined F1: {val_results['eval_f1_combined']:.4f}")
    
    # Get OOF predictions on validation fold
    print(f"\nGetting OOF predictions for fold {fold}...")
    oof_output = trainer.predict(fold_val_dataset)
    logits_clarity_oof, logits_evasion_oof = oof_output.predictions

    # Store out-of-fold predictions
    oof_preds_clarity[val_idx] = np.argmax(logits_clarity_oof, axis=-1)
    oof_preds_evasion[val_idx] = np.argmax(logits_evasion_oof, axis=-1)
    
    # Get true labels for this fold
    fold_val_df = train_df.iloc[val_idx]
    oof_true_clarity[val_idx] = [clarity_label2id[label] for label in fold_val_df['clarity_label']]
    oof_true_evasion[val_idx] = [evasion_label2id[label] for label in fold_val_df['evasion_label']]
    
    del model, trainer
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("CROSS-VALIDATION COMPLETE")
print(f"{'='*60}")

## Overall Out-of-Fold Results

In [ ]:
# Overall CV metrics
overall_acc_clarity = accuracy_score(oof_true_clarity, oof_preds_clarity)
overall_f1_clarity = f1_score(oof_true_clarity, oof_preds_clarity, average='macro')
overall_acc_evasion = accuracy_score(oof_true_evasion, oof_preds_evasion)
overall_f1_evasion = f1_score(oof_true_evasion, oof_preds_evasion, average='macro')

print("\n" + "="*60)
print("OVERALL OUT-OF-FOLD RESULTS")
print("="*60)
print(f"\nClarity (Subtask 1):")
print(f"  Accuracy: {overall_acc_clarity:.4f}")
print(f"  Macro F1: {overall_f1_clarity:.4f}")

print(f"\nEvasion (Subtask 2):")
print(f"  Accuracy: {overall_acc_evasion:.4f}")
print(f"  Macro F1: {overall_f1_evasion:.4f}")

print(f"\nCombined F1: {(overall_f1_clarity + overall_f1_evasion) / 2:.4f}")

## Detailed Classification Reports

In [ ]:
# Convert predictions to label strings
oof_pred_labels_clarity = [clarity_id2label[p] for p in oof_preds_clarity]
oof_true_labels_clarity = [clarity_id2label[t] for t in oof_true_clarity]
oof_pred_labels_evasion = [evasion_id2label[p] for p in oof_preds_evasion]
oof_true_labels_evasion = [evasion_id2label[t] for t in oof_true_evasion]

print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORTS (OUT-OF-FOLD)")
print("="*60)

print("\n--- CLARITY (Subtask 1) ---")
print(classification_report(oof_true_labels_clarity, oof_pred_labels_clarity, labels=clarity_labels, zero_division=0))

print("\n--- EVASION (Subtask 2) ---")
print(classification_report(oof_true_labels_evasion, oof_pred_labels_evasion, labels=evasion_classes, zero_division=0))

## Confusion Matrix Helper Functions

In [ ]:
def plot_confusion_matrix(cm, labels, title, filename, normalize=False):
    """Plot and save confusion matrix"""
    if normalize:
        cm_display = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.3f'
        cmap = 'Blues'
    else:
        cm_display = cm
        fmt = 'd'
        cmap = 'Blues'
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_display, annot=True, fmt=fmt, cmap=cmap, 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Normalized Count' if normalize else 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")
    plt.show()

def cm_to_latex(cm, labels, title):
    """Convert confusion matrix to LaTeX table format"""
    n = len(labels)
    
    # Start table
    latex = "\\begin{table}[h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{title}}}\n"
    latex += "\\begin{tabular}{l|" + "c" * n + "}\n"
    latex += "\\hline\n"
    
    # Header row
    latex += " & " + " & ".join([f"\\textbf{{{label}}}" for label in labels]) + " \\\\\n"
    latex += "\\hline\n"
    
    # Data rows
    for i, label in enumerate(labels):
        row = f"\\textbf{{{label}}}"
        for j in range(n):
            row += f" & {cm[i, j]}"
        row += " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

def cm_to_latex_normalized(cm, labels, title):
    """Convert normalized confusion matrix to LaTeX table format"""
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    n = len(labels)
    
    latex = "\\begin{table}[h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{title}}}\n"
    latex += "\\begin{tabular}{l|" + "c" * n + "}\n"
    latex += "\\hline\n"
    
    # Header row
    latex += " & " + " & ".join([f"\\textbf{{{label}}}" for label in labels]) + " \\\\\n"
    latex += "\\hline\n"
    
    # Data rows
    for i, label in enumerate(labels):
        row = f"\\textbf{{{label}}}"
        for j in range(n):
            row += f" & {cm_norm[i, j]:.3f}"
        row += " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

print("Helper functions defined for confusion matrix visualization and LaTeX export")

## Clarity Confusion Matrix

In [ ]:
print("\n" + "="*60)
print("SUBTASK 1: CLARITY CONFUSION MATRIX (OUT-OF-FOLD)")
print("="*60)

cm_clarity = confusion_matrix(oof_true_labels_clarity, oof_pred_labels_clarity, labels=clarity_labels)

print("\n--- Raw Confusion Matrix ---")
print(cm_clarity)

# Plot raw CM
plot_confusion_matrix(
    cm_clarity, 
    clarity_labels,
    "Clarity Confusion Matrix - 3 Fold (Raw Counts)",
    "./confusion_matrices_3fold/clarity_raw.png",
    normalize=False
)

# Plot normalized CM
plot_confusion_matrix(
    cm_clarity, 
    clarity_labels,
    "Clarity Confusion Matrix - 3 Fold (Normalized)",
    "./confusion_matrices_3fold/clarity_normalized.png",
    normalize=True
)

# LaTeX tables
print("\n--- LaTeX Table (Raw) ---")
latex_clarity = cm_to_latex(cm_clarity, clarity_labels, "Clarity Confusion Matrix (3-Fold)")
print(latex_clarity)

print("\n--- LaTeX Table (Normalized) ---")
latex_clarity_norm = cm_to_latex_normalized(cm_clarity, clarity_labels, "Clarity Confusion Matrix Normalized (3-Fold)")
print(latex_clarity_norm)

## Evasion Confusion Matrix

In [ ]:
print("\n" + "="*60)
print("SUBTASK 2: EVASION CONFUSION MATRIX (OUT-OF-FOLD)")
print("="*60)

cm_evasion = confusion_matrix(oof_true_labels_evasion, oof_pred_labels_evasion, labels=evasion_classes)

print("\n--- Raw Confusion Matrix ---")
print(cm_evasion)

# Plot raw CM
plot_confusion_matrix(
    cm_evasion, 
    evasion_classes,
    "Evasion Confusion Matrix - 3 Fold (Raw Counts)",
    "./confusion_matrices_3fold/evasion_raw.png",
    normalize=False
)

# Plot normalized CM
plot_confusion_matrix(
    cm_evasion, 
    evasion_classes,
    "Evasion Confusion Matrix - 3 Fold (Normalized)",
    "./confusion_matrices_3fold/evasion_normalized.png",
    normalize=True
)

# LaTeX tables
print("\n--- LaTeX Table (Raw) ---")
latex_evasion = cm_to_latex(cm_evasion, evasion_classes, "Evasion Confusion Matrix (3-Fold)")
print(latex_evasion)

print("\n--- LaTeX Table (Normalized) ---")
latex_evasion_norm = cm_to_latex_normalized(cm_evasion, evasion_classes, "Evasion Confusion Matrix Normalized (3-Fold)")
print(latex_evasion_norm)

print("\n" + "="*60)
print("All confusion matrices saved to ./confusion_matrices_3fold/")
print("="*60)

## Per-Fold Statistics (Mean ± Std)

In [ ]:
print("="*60)
print("PER-FOLD PERFORMANCE STATISTICS")
print("="*60)

metrics_df = pd.DataFrame(fold_metrics)

print(f"\n{'Fold':<6} {'Clarity Acc':>12} {'Clarity F1':>12} {'Evasion Acc':>12} {'Evasion F1':>12} {'Combined F1':>12}")
print("-" * 78)
for _, row in metrics_df.iterrows():
    print(f"{int(row['fold']):<6} {row['val_accuracy_clarity']:>12.4f} {row['val_f1_clarity']:>12.4f} "
          f"{row['val_accuracy_evasion']:>12.4f} {row['val_f1_evasion']:>12.4f} "
          f"{row['val_f1_combined']:>12.4f}")
print("-" * 78)
print(f"{'Mean':<6} {metrics_df['val_accuracy_clarity'].mean():>12.4f} "
      f"{metrics_df['val_f1_clarity'].mean():>12.4f} "
      f"{metrics_df['val_accuracy_evasion'].mean():>12.4f} "
      f"{metrics_df['val_f1_evasion'].mean():>12.4f} "
      f"{metrics_df['val_f1_combined'].mean():>12.4f}")
print(f"{'Std':<6} {metrics_df['val_accuracy_clarity'].std():>12.4f} "
      f"{metrics_df['val_f1_clarity'].std():>12.4f} "
      f"{metrics_df['val_accuracy_evasion'].std():>12.4f} "
      f"{metrics_df['val_f1_evasion'].std():>12.4f} "
      f"{metrics_df['val_f1_combined'].std():>12.4f}")

print("\n" + "="*60)
print("SUMMARY: MEAN ± STD ACROSS FOLDS")
print("="*60)

print(f"\nClarity (Subtask 1):")
print(f"  Accuracy: {metrics_df['val_accuracy_clarity'].mean():.4f} ± {metrics_df['val_accuracy_clarity'].std():.4f}")
print(f"  Macro F1: {metrics_df['val_f1_clarity'].mean():.4f} ± {metrics_df['val_f1_clarity'].std():.4f}")

print(f"\nEvasion (Subtask 2):")
print(f"  Accuracy: {metrics_df['val_accuracy_evasion'].mean():.4f} ± {metrics_df['val_accuracy_evasion'].std():.4f}")
print(f"  Macro F1: {metrics_df['val_f1_evasion'].mean():.4f} ± {metrics_df['val_f1_evasion'].std():.4f}")

print(f"\nCombined:")
print(f"  F1 Score: {metrics_df['val_f1_combined'].mean():.4f} ± {metrics_df['val_f1_combined'].std():.4f}")

## LaTeX Results Table

In [ ]:
print("\n" + "="*60)
print("LATEX TABLE FOR PAPER")
print("="*60)

latex_results = """
\\begin{table}[h]
\\centering
\\caption{3-Fold Cross-Validation Results - Max Pooling}
\\begin{tabular}{lcccc}
\\hline
\\textbf{Metric} & \\textbf{Clarity Acc} & \\textbf{Clarity F1} & \\textbf{Evasion Acc} & \\textbf{Evasion F1} \\\\
\\hline
"""

for _, row in metrics_df.iterrows():
    latex_results += f"Fold {int(row['fold'])} & {row['val_accuracy_clarity']:.4f} & {row['val_f1_clarity']:.4f} & "
    latex_results += f"{row['val_accuracy_evasion']:.4f} & {row['val_f1_evasion']:.4f} \\\\\n"

latex_results += "\\hline\n"
latex_results += f"Mean & {metrics_df['val_accuracy_clarity'].mean():.4f} & {metrics_df['val_f1_clarity'].mean():.4f} & "
latex_results += f"{metrics_df['val_accuracy_evasion'].mean():.4f} & {metrics_df['val_f1_evasion'].mean():.4f} \\\\\n"
latex_results += f"Std & {metrics_df['val_accuracy_clarity'].std():.4f} & {metrics_df['val_f1_clarity'].std():.4f} & "
latex_results += f"{metrics_df['val_accuracy_evasion'].std():.4f} & {metrics_df['val_f1_evasion'].std():.4f} \\\\\n"
latex_results += """\\hline
\\end{tabular}
\\end{table}
"""

print(latex_results)